# PCA in Practice with MNIST

## Focus
In this activity, you will explore Principal Component Analysis (PCA) on the MNIST dataset.

Your goals are to:

Understand how PCA works in practice

Explore the impact of dimensionality reduction on training time and model performance

Reflect on when PCA is effective in machine learning workflows

## Dataset
MNIST dataset of handwritten digits

Training set: first 60,000 images

Test set: remaining 10,000 images

## Part 1 — Implementation

### Step 1 — Baseline Random Forest

1. Load the MNIST dataset and split into training and test sets.

In [53]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist["data"], mnist["target"].astype(int)

X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

In [54]:
# applying standardScaleer
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

2. Train a Random Forest classifier on the full dataset.

In [55]:
from sklearn.ensemble import RandomForestClassifier
import time
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
start = time.perf_counter()
rf_clf.fit(X_train, y_train)
rf_clf_training_time = time.perf_counter() - start

3. Record the training time.

In [56]:
print("Random Forest Classifier training time:", round(rf_clf_training_time, 1))

Random Forest Classifier training time: 4.0


4. Evaluate the classifier on the test set (accuracy, confusion matrix, or other relevant metrics)

In [57]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = rf_clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification report:")
print(classification_report(y_test, y_pred))

Test accuracy: 0.9704
Confusion matrix:
[[ 971    0    0    0    0    2    3    1    3    0]
 [   0 1127    2    2    0    1    2    0    1    0]
 [   6    0 1002    5    3    0    3    8    5    0]
 [   1    0    9  972    0    9    0    9    8    2]
 [   1    0    0    0  955    0    5    1    4   16]
 [   5    1    1    9    2  860    5    2    5    2]
 [   7    3    0    0    3    4  936    0    5    0]
 [   1    4   20    2    0    0    0  990    2    9]
 [   5    0    6    6    5    5    5    4  930    8]
 [   7    6    2   12   12    1    0    4    4  961]]
Classification report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       980
           1       0.99      0.99      0.99      1135
           2       0.96      0.97      0.97      1032
           3       0.96      0.96      0.96      1010
           4       0.97      0.97      0.97       982
           5       0.98      0.96      0.97       892
           6       0.98    

### Step 2 — Apply PCA

1. Use PCA to reduce the dataset’s dimensionality, keeping 95% explained variance

In [58]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("Original n_features:", X_train.shape[1], "Reduced:", X_train_pca.shape[1])

Original n_features: 784 Reduced: 331


2. Train a new Random Forest classifier on the reduced dataset.

In [59]:
rf_clf_pca = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
start = time.perf_counter()
rf_clf_pca.fit(X_train_pca, y_train)
rf_clf_pca_training_time = time.perf_counter() - start

3. Record the training time

In [60]:
print("Random Forest Classifier with PCA training time:", round(rf_clf_pca_training_time, 1))

Random Forest Classifier with PCA training time: 15.8


4. Evaluate the classifier on the test set.

In [61]:
y_pred = rf_clf_pca.predict(X_test_pca)

acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification report:")
print(classification_report(y_test, y_pred))

Test accuracy: 0.9369
Confusion matrix:
[[ 954    0    7    2    0    5    9    2    1    0]
 [   0 1118    4    2    2    1    6    0    2    0]
 [   8    0  957   21    3    1    6   11   24    1]
 [   2    1   16  938    1   12    1   15   18    6]
 [   1    0    8    2  929    1    9    4    3   25]
 [   6    0    4   36    5  807   13    5    9    7]
 [  11    2    6    0    5   14  918    0    2    0]
 [   1    7   21    4    9    0    0  956    2   28]
 [   8    0    9   26    7   19    7    7  886    5]
 [   6    5    5   15   33    6    0   30    3  906]]
Classification report:
              precision    recall  f1-score   support

           0       0.96      0.97      0.97       980
           1       0.99      0.99      0.99      1135
           2       0.92      0.93      0.93      1032
           3       0.90      0.93      0.91      1010
           4       0.93      0.95      0.94       982
           5       0.93      0.90      0.92       892
           6       0.95    

### Questions to Consider:
- Was training significantly faster?

No, for Random Forestm, training took longer with PCA, from 3.7s with baseline to 10.2 with PCA 

- How did the model’s performance compare to the baseline?

With the performance, with model's test accuracy dropped with PCA, from 97.05% with baseline to 94.81% with PCA

### Step 3 — Try with SGDClassifier

1. Train an SGDClassifier on the full dataset and record training time and performance.

In [62]:
from sklearn.linear_model import SGDClassifier

sgd_clf = SGDClassifier(random_state=42, max_iter=1000, tol=1e-3)

start = time.perf_counter()
sgd_clf.fit(X_train, y_train)
sgd_training_time = time.perf_counter() - start

y_pred = sgd_clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")
print(f"Training time: {sgd_training_time:.2f} seconds")

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification report:")
print(classification_report(y_test, y_pred))

Test accuracy: 0.8933
Training time: 239.63 seconds
Confusion matrix:
[[ 937    0    0    0    0    3    4    1   35    0]
 [   0 1083    5    1    0    3    4    0   39    0]
 [   4    3  890   14    7    2   13    6   89    4]
 [   4    0   13  874    0   20    2    7   82    8]
 [   1    0    6    0  877    1    6    4   66   21]
 [   5    2    1   31    8  716   17    7   98    7]
 [  10    2   10    0    8   13  881    1   33    0]
 [   2    2   16    5    5    1    0  924   49   24]
 [   6    4    4   15    3   24    8    1  905    4]
 [   5    5    0    6   24    4    0   21   98  846]]
Classification report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96       980
           1       0.98      0.95      0.97      1135
           2       0.94      0.86      0.90      1032
           3       0.92      0.87      0.89      1010
           4       0.94      0.89      0.92       982
           5       0.91      0.80      0.85       8

2. Train the same classifier on the PCA-reduced dataset.

In [63]:
sgd_clf_pca = SGDClassifier(random_state=42, max_iter=1000, tol=1e-3)

start = time.perf_counter()
sgd_clf_pca.fit(X_train_pca, y_train)
sgd_pca_training_time = time.perf_counter() - start

y_pred_pca = sgd_clf_pca.predict(X_test_pca)

acc_pca = accuracy_score(y_test, y_pred_pca)
print(f"Test accuracy (PCA): {acc_pca:.4f}")
print(f"Training time (PCA): {sgd_pca_training_time:.2f} seconds")

print("Confusion matrix (PCA):")
print(confusion_matrix(y_test, y_pred_pca))

print("Classification report (PCA):")
print(classification_report(y_test, y_pred_pca))

Test accuracy (PCA): 0.8959
Training time (PCA): 125.85 seconds
Confusion matrix (PCA):
[[ 938    0    0    0    0    3    5    1   33    0]
 [   0 1083    4    1    0    3    4    0   40    0]
 [   3    2  894   14    8    2   11    6   89    3]
 [   4    0   11  880    0   23    2    6   76    8]
 [   1    0    6    0  871    1    7    3   67   26]
 [   5    2    0   29    9  726   17    6   90    8]
 [   8    2   10    1    7   13  885    1   31    0]
 [   2    2   15    5    5    1    0  925   51   22]
 [   5    4    4   17    3   24    7    2  903    5]
 [   5    5    1    7   23    4    0   21   89  854]]
Classification report (PCA):
              precision    recall  f1-score   support

           0       0.97      0.96      0.96       980
           1       0.98      0.95      0.97      1135
           2       0.95      0.87      0.90      1032
           3       0.92      0.87      0.90      1010
           4       0.94      0.89      0.91       982
           5       0.91    

3. Compare results.

In [64]:
rf_acc = accuracy_score(y_test, rf_clf.predict(X_test))
rf_pca_acc = accuracy_score(y_test, rf_clf_pca.predict(X_test_pca))

sgd_acc = accuracy_score(y_test, sgd_clf.predict(X_test))
sgd_pca_acc = accuracy_score(y_test, sgd_clf_pca.predict(X_test_pca))

print("Random Forest (full)\t- acc: {:.4f}, train time: {:.2f}s".format(rf_acc, rf_clf_training_time))
print("Random Forest (PCA)\t- acc: {:.4f}, train time: {:.2f}s".format(rf_pca_acc, rf_clf_pca_training_time))
print()
print("SGDClassifier (full)\t- acc: {:.4f}, train time: {:.2f}s".format(sgd_acc, sgd_training_time))
print("SGDClassifier (PCA)\t- acc: {:.4f}, train time: {:.2f}s".format(sgd_pca_acc, sgd_pca_training_time))

Random Forest (full)	- acc: 0.9704, train time: 3.95s
Random Forest (PCA)	- acc: 0.9369, train time: 15.84s

SGDClassifier (full)	- acc: 0.8933, train time: 239.63s
SGDClassifier (PCA)	- acc: 0.8959, train time: 125.85s


old results without StandardScaler:

Random Forest (full)	- acc: 0.9705, train time: 3.68s
Random Forest (PCA)	- acc: 0.9481, train time: 10.24s

SGDClassifier (full)	- acc: 0.8740, train time: 124.15s
SGDClassifier (PCA)	- acc: 0.8959, train time: 24.36s

In [65]:
# training XGBoost
import xgboost as xgb
from xgboost.sklearn import XGBClassifier
xgb_clf = XGBClassifier(objective='binary:logistic', n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)


ModuleNotFoundError: No module named 'xgboost'

### Questions to Consider:
- How much does PCA help when using SGDClassifier compared to Random Forest?

In comparison with Random Forestt and SGDClassifier, SGDClassifier did alot better using PCA than Random Forest,
SGDClassifier was able to improve in both performnce and training time with 87.4% baseline test accuracy to 89.59% PCA test accuracy and 124.15s baseline train time to 24.36s PCA train time

- Why might the effect differ between model types?

SGD is a linear model, so reducing noisy/redundant features with PCA can make optimization easier and faster. Random Forest already handles high-dimensional raw features well via tree splits and feature subsampling, while PCA can remove/blur useful nonlinear structure, so it may hurt accuracy

## Part 2 — Reflection & Summary
Write a short reflection (1–2 paragraphs) addressing:
1. What did you learn about PCA in practice?

- Consider effects on training time, model performance, and data dimensionality.

2. Any observations or surprises?

- Did PCA help some models more than others?

- How did dimensionality reduction affect accuracy

I learned that with applying PCA, you treat it like StandardScaler() to your training and test sets. It basically a good practice technique to help 'spcifc models' improve in both time and performnace. PCA should only be used in spcfic cases, 1, when you have large amount of features, with PCA you are able to reduce a large feature set to features that will help a model, basaiicaly get ride of noice. with PCA applied on this data, it went from 784 featurees to 154 being used. This just makes training in gereral more efficient.

One thing that i was surprised about was the amount of features taken out with PCA and how much time was saved with PCA. When applied correctly, PCA was able to reduce training time by around 100s while still improving test accuracy. Usally i would think that a model that needs to train longer, mean i will proabally have a high accuracy. But that was before taking this course. a models perfomance is really dependent on other things like the math behind it and other things. With the amount of feature reduced down by over 500 features, Im surprised that the accuracy improved rather than got close to the base line. 